In [1]:
import copy
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

ROOT = Path(".")
ANNOTATION_PATH = ROOT / "multi_label_annotations.json"
CHECKPOINT_INIT_PATH = ROOT / "best_densenet121_scale224.pth"
OUTPUT_CKPT_PATH = ROOT / "best_densenet121_multilabel_scale224.pth"

BATCH_SIZE = 32
if os.name == "nt":
    NUM_WORKERS = 0
else:
    NUM_WORKERS = min(4, os.cpu_count() or 2)
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

WARMUP_EPOCHS = 3
FINETUNE_EPOCHS = 15
VAL_RATIO = 0.2
RANDOM_STATE = 7
THRESH_GRID = np.arange(0.10, 0.91, 0.05)

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

USE_AMP = torch.cuda.is_available()
USE_CHANNELS_LAST = torch.cuda.is_available()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("Using device:", device)
print("Annotation path:", ANNOTATION_PATH)
print("Checkpoint init path:", CHECKPOINT_INIT_PATH)
print(f"DataLoader workers: {NUM_WORKERS} | persistent: {PERSISTENT_WORKERS} | prefetch: {PREFETCH_FACTOR}")
print(f"AMP enabled: {USE_AMP} | channels_last: {USE_CHANNELS_LAST}")

assert ANNOTATION_PATH.exists(), f"Missing annotation file: {ANNOTATION_PATH}"
assert CHECKPOINT_INIT_PATH.exists(), f"Missing checkpoint file: {CHECKPOINT_INIT_PATH}"

Using device: cuda
Annotation path: multi_label_annotations.json
Checkpoint init path: best_densenet121_scale224.pth
DataLoader workers: 0 | persistent: False | prefetch: None
AMP enabled: True | channels_last: True


In [2]:
def seed_everything(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_multilabel_classes(anno: dict):
    det_map = anno.get("detector_label_map", {})
    if det_map:
        ordered = sorted(det_map.items(), key=lambda kv: kv[1])
        return [name for name, _ in ordered]

    cls_names = anno.get("classifier_class_names", [])
    return [c for c in cls_names if c != "notumor"]


with open(ANNOTATION_PATH, "r", encoding="utf-8") as f:
    anno = json.load(f)

records = anno["images"]
ml_classes = build_multilabel_classes(anno)
class_to_idx = {c: i for i, c in enumerate(ml_classes)}
num_classes = len(ml_classes)

print("Multi-label classes:", ml_classes)
print("Number of samples:", len(records))

def build_target_vector(active_classes):
    y = np.zeros(num_classes, dtype=np.float32)
    for cls_name in active_classes:
        if cls_name in class_to_idx:
            y[class_to_idx[cls_name]] = 1.0
    return y

targets_all = np.stack([build_target_vector(r.get("active_classes", [])) for r in records])
strat_keys = np.array(["".join(row.astype(np.int32).astype(str).tolist()) for row in targets_all])

indices = np.arange(len(records))
try:
    train_idx, val_idx = train_test_split(
        indices,
        test_size=VAL_RATIO,
        random_state=RANDOM_STATE,
        stratify=strat_keys,
    )
    print("Split mode: stratified by multi-hot key")
except ValueError:
    train_idx, val_idx = train_test_split(
        indices,
        test_size=VAL_RATIO,
        random_state=RANDOM_STATE,
        shuffle=True,
    )
    print("Split mode: random fallback (insufficient class-combination counts)")

print(f"Train size: {len(train_idx)} | Val size: {len(val_idx)}")
label_cardinality = targets_all.sum(axis=1)
print("Mean labels/image:", float(label_cardinality.mean()))
print("Images with 0 labels:", int((label_cardinality == 0).sum()))
print("Images with 1 labels:", int((label_cardinality == 1).sum()))
print("Images with 2+ labels:", int((label_cardinality >= 2).sum()))

Multi-label classes: ['glioma', 'meningioma', 'pituitary']
Number of samples: 17813
Split mode: random fallback (insufficient class-combination counts)
Train size: 14250 | Val size: 3563
Mean labels/image: 1.579857349395752
Images with 0 labels: 3847
Images with 1 labels: 2810
Images with 2+ labels: 11156


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

class MultiLabelTumorDataset(Dataset):
    def __init__(self, records, root, class_to_idx, transform=None, cache_images=False):
        self.records = records
        self.root = root
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.num_classes = len(class_to_idx)
        self.cache_images = cache_images
        self._cache = {}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = self.root / rec["image_path"]
        cache_key = str(img_path)

        if self.cache_images and cache_key in self._cache:
            image = self._cache[cache_key].copy()
        else:
            with Image.open(img_path) as img:
                image = img.convert("RGB")
            if self.cache_images:
                self._cache[cache_key] = image

        if self.transform is not None:
            image = self.transform(image)

        target = torch.zeros(self.num_classes, dtype=torch.float32)
        for cls_name in rec.get("active_classes", []):
            if cls_name in self.class_to_idx:
                target[self.class_to_idx[cls_name]] = 1.0

        return image, target


train_records = [records[i] for i in train_idx]
val_records = [records[i] for i in val_idx]

train_dataset = MultiLabelTumorDataset(train_records, ROOT, class_to_idx, transform=train_transform, cache_images=False)
val_dataset = MultiLabelTumorDataset(val_records, ROOT, class_to_idx, transform=val_transform, cache_images=True)

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": PERSISTENT_WORKERS,
}
if PREFETCH_FACTOR is not None:
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

targets_train = np.stack([build_target_vector(r.get("active_classes", [])) for r in train_records])
pos_count = targets_train.sum(axis=0)
neg_count = len(targets_train) - pos_count
pos_weight = (neg_count + 1e-6) / (pos_count + 1e-6)
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device=device)

print("Per-class positive counts:", {c: int(pos_count[i]) for i, c in enumerate(ml_classes)})
print("Per-class pos_weight:", {c: float(np.round(pos_weight[i], 3)) for i, c in enumerate(ml_classes)})

Per-class positive counts: {'glioma': 11129, 'meningioma': 4358, 'pituitary': 7032}
Per-class pos_weight: {'glioma': 0.2800000011920929, 'meningioma': 2.2699999809265137, 'pituitary': 1.0260000228881836}


In [4]:
def build_model(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 128),
        nn.ReLU(inplace=True),
        nn.Linear(128, num_classes),
    )
    return model


def load_backbone_from_checkpoint(model, ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    cleaned_state = {k.replace("module.", ""): v for k, v in state_dict.items()}

    model_state = model.state_dict()
    compatible = {}
    for k, v in cleaned_state.items():
        if k in model_state and model_state[k].shape == v.shape:
            compatible[k] = v

    model_state.update(compatible)
    model.load_state_dict(model_state)
    print(f"Loaded {len(compatible)} compatible tensors from checkpoint.")
    return model


model = build_model(num_classes=num_classes)
model = load_backbone_from_checkpoint(model, CHECKPOINT_INIT_PATH)

for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)
if USE_CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
if torch.cuda.is_available():
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
else:
    scaler = None


def make_optimizer_head(model, lr=2e-3):
    return optim.AdamW(model.classifier.parameters(), lr=lr, weight_decay=1e-4)


def make_optimizer_finetune(model, lr=5e-4):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    return optim.AdamW(trainable_params, lr=lr, weight_decay=1e-4)

Loaded 727 compatible tensors from checkpoint.


In [5]:
@torch.no_grad()
def run_validation(model, loader, criterion, device, thresholds=0.5, use_amp=False, channels_last=False):
    model.eval()
    running_loss = 0.0

    all_probs = []
    all_targets = []

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    autocast_dtype = torch.float16 if autocast_device == "cuda" else torch.bfloat16

    for images, targets in loader:
        if channels_last:
            images = images.contiguous(memory_format=torch.channels_last)
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type=autocast_device, dtype=autocast_dtype, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, targets)
        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    y_prob = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_targets, axis=0).astype(np.int32)

    thr = np.array(thresholds, dtype=np.float32)
    if thr.ndim == 0:
        thr = np.full(y_true.shape[1], float(thr), dtype=np.float32)

    y_pred = (y_prob >= thr.reshape(1, -1)).astype(np.int32)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return val_loss, micro_f1, macro_f1, y_true, y_prob


def train_one_epoch(model, loader, criterion, optimizer, device, use_amp=False, scaler=None, channels_last=False):
    model.train()
    running_loss = 0.0

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    autocast_dtype = torch.float16 if autocast_device == "cuda" else torch.bfloat16

    for images, targets in loader:
        if channels_last:
            images = images.contiguous(memory_format=torch.channels_last)
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=autocast_device, dtype=autocast_dtype, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, targets)

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


def tune_per_class_thresholds(y_true, y_prob, grid):
    best_thr = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    best_f1 = np.zeros(y_true.shape[1], dtype=np.float32)

    for c in range(y_true.shape[1]):
        gt = y_true[:, c]
        prob = y_prob[:, c]

        chosen_thr = 0.5
        chosen_f1 = -1.0
        for t in grid:
            pred = (prob >= t).astype(np.int32)
            f1 = f1_score(gt, pred, zero_division=0)
            if f1 > chosen_f1:
                chosen_f1 = f1
                chosen_thr = float(t)

        best_thr[c] = chosen_thr
        best_f1[c] = chosen_f1

    return best_thr, best_f1

In [6]:
seed_everything(RANDOM_STATE)

best_state = copy.deepcopy(model.state_dict())
best_val_loss = float("inf")
best_epoch = -1
best_thresholds = np.full(num_classes, 0.5, dtype=np.float32)

patience = 5
patience_counter = 0
global_epoch = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_micro_f1": [],
    "val_macro_f1": [],
}

optimizer = make_optimizer_head(model, lr=2e-3)
for epoch in range(WARMUP_EPOCHS):
    epoch_start = time.perf_counter()
    global_epoch += 1
    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        use_amp=USE_AMP,
        scaler=scaler,
        channels_last=USE_CHANNELS_LAST,
    )
    val_loss, micro_f1, macro_f1, y_true, y_prob = run_validation(
        model,
        val_loader,
        criterion,
        device,
        thresholds=0.5,
        use_amp=USE_AMP,
        channels_last=USE_CHANNELS_LAST,
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_micro_f1"].append(micro_f1)
    history["val_macro_f1"].append(macro_f1)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = global_epoch
        best_state = copy.deepcopy(model.state_dict())
        best_thresholds, _ = tune_per_class_thresholds(y_true, y_prob, THRESH_GRID)
        patience_counter = 0
    else:
        patience_counter += 1

    epoch_time = time.perf_counter() - epoch_start
    print(
        f"[Warmup {epoch + 1}/{WARMUP_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val micro-F1: {micro_f1:.4f} | Val macro-F1: {macro_f1:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

for param in model.features.denseblock4.parameters():
    param.requires_grad = True
for param in model.features.norm5.parameters():
    param.requires_grad = True

optimizer = make_optimizer_finetune(model, lr=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6)

for epoch in range(FINETUNE_EPOCHS):
    epoch_start = time.perf_counter()
    global_epoch += 1
    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        use_amp=USE_AMP,
        scaler=scaler,
        channels_last=USE_CHANNELS_LAST,
    )
    val_loss, micro_f1, macro_f1, y_true, y_prob = run_validation(
        model,
        val_loader,
        criterion,
        device,
        thresholds=0.5,
        use_amp=USE_AMP,
        channels_last=USE_CHANNELS_LAST,
    )
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_micro_f1"].append(micro_f1)
    history["val_macro_f1"].append(macro_f1)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = global_epoch
        best_state = copy.deepcopy(model.state_dict())
        best_thresholds, _ = tune_per_class_thresholds(y_true, y_prob, THRESH_GRID)
        patience_counter = 0
    else:
        patience_counter += 1

    epoch_time = time.perf_counter() - epoch_start
    print(
        f"[Finetune {epoch + 1}/{FINETUNE_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val micro-F1: {micro_f1:.4f} | Val macro-F1: {macro_f1:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_state)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "multilabel_class_names": ml_classes,
        "thresholds": best_thresholds.tolist(),
        "best_val_loss": float(best_val_loss),
        "best_epoch": int(best_epoch),
        "init_mode": "checkpoint_backbone",
    },
    OUTPUT_CKPT_PATH,
 )

print(f"Saved best checkpoint to: {OUTPUT_CKPT_PATH.resolve()}")
print("Best thresholds:", {c: float(np.round(t, 3)) for c, t in zip(ml_classes, best_thresholds)})

KeyboardInterrupt: 

In [ ]:
model.eval()
val_loss, micro_f1, macro_f1, y_true, y_prob = run_validation(
    model,
    val_loader,
    criterion,
    device,
    thresholds=best_thresholds,
    use_amp=USE_AMP,
    channels_last=USE_CHANNELS_LAST,
)

y_pred = (y_prob >= best_thresholds.reshape(1, -1)).astype(np.int32)

print(f"Final Val Loss: {val_loss:.4f}")
print(f"Final Val micro-F1: {micro_f1:.4f}")
print(f"Final Val macro-F1: {macro_f1:.4f}")

for i, cls_name in enumerate(ml_classes):
    report = classification_report(
        y_true[:, i],
        y_pred[:, i],
        labels=[0, 1],
        target_names=["neg", "pos"],
        zero_division=0,
        digits=4,
    )
    print(f"\n Class: {cls_name} | Threshold: {best_thresholds[i]:.2f}")
    print(report)